# Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.impute import SimpleImputer

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline

import xgboost as xgb

In [ ]:
df = pd.read_csv('../data/df_stats.csv', sep=';', dtype={'Code_INSEE': str})
df_train = df[df['Année'] == 2022].copy()
display(df_train.head(5))
print(f"Dataset chargé : {df_train.shape[0]} lignes, {df_train.shape[1]} colonnes")

print(f"\nDistribution de la cible (vote politique) :")
print(df_train['Résultat'].value_counts())
print(f"\nProportions :")
print(df_train['Résultat'].value_counts(normalize=True).round(4) * 100)

print('\nColonnes :')
print(df_train.columns)

# df_null = df[df.isna().any(axis=1)]
# df = df.dropna()
# print(f"Valeurs manquantes supprimées, df final : {df.shape[0]} lignes, {df.shape[1]} colonnes")

# Feature engineering

In [ ]:
df_train = df_train.drop(columns=['Année','Libellé de la commune', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', '% gauche/Exp', '% centre/Exp', '% droite/Exp'])


def convert_columns_to_percentages(df, list_columns, divider_column):
    for col in list_columns:
        df[col] = df[col] / df[divider_column] * 100
    return df


# Transformation des genres, catégories socio-professionnelles, tranches d'âge et statuts maritaux en pourcentages de la population active
columns_to_convert = [
    'Hommes', 
    'Femmes', 
    'Agriculteurs', 
    'Artisans', 
    'Cadres', 
    'Intermédiaires', 
    'Employés', 
    'Ouvriers',
    'Retraités',
    'Etudiants',
    'Inactifs',
    '15-24 ans',
    '25-39 ans',
    '40-54 ans',
    '55-64 ans',
    '65-79 ans',
    '80 ans et +',
    'Mariés',
    'Pacsés',
    'Concubinage',
    'Veufs',
    'Divorcés',
    'Célibataires',
]

df_train = convert_columns_to_percentages(df_train, columns_to_convert, 'Population_active')



# Transformation des compositions de ménages en pourcentage de la population avec enfants

columns_to_convert_household = [
    'Personne seule',
    'Homme seul',
    'Femme seule',
    'Colocation',
    'Famille',
    'Famille monoparentale',
    'Couple sans enfant',
    'Couple avec enfants',
]
df_train = convert_columns_to_percentages(df_train, columns_to_convert_household, 'Population avec enfants')

display(df_train)



In [ ]:
# On vérifie que les pourcentages sont corrects

df_test = df_train.copy()
df_test = df_test.drop(columns=['Code_INSEE', 'Résultat', 'Population_active', 'Population avec enfants'])

mask_anomalies = (df_test < 0) | (df_test > 100)
df_anomalies = df_test[mask_anomalies.any(axis=1)]

display(df_anomalies)

In [ ]:
# On vérifie les doublons et valeurs manquantes
duplicates_count = df_train.duplicated().sum()
print('Nombre de doublons :', duplicates_count)

null_values_count = df_train.isnull().sum()
print(f'\nNombre de valeurs manquantes : {null_values_count}')

In [ ]:
# On exporte le df en csv
df_train.to_csv('df_model.csv', index=False, sep=';', encoding='utf-8')

# Modélisation

## Split test-train

In [ ]:
X = df_train.drop(columns=['Code_INSEE', 'Résultat'])
y = df_train['Résultat']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("✓ Données préparées")
print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")

## Fonctions

In [ ]:
groups = ['centre', 'droite', 'gauche']


def train_model_and_detect_overfitting(model, y_train, y_test):
    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)


    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    gap = abs(train_score - test_score)


    print(f"  Train : {train_score:.2%}")
    print(f"  Test  : {test_score:.2%}")
    print(f"  Gap   : {gap:.2%}\n")


    if gap > 0.10:
        print("⚠️ OVERFITTING détecté !")
    elif train_score < 0.75:
        print("⚠️ UNDERFITTING détecté !")
    else:
        print("✅ Bon équilibre\n")
    
    return y_pred_test, y_pred_train


def display_confusion_matrix(y, y_pred):
    matrix = confusion_matrix(y, y_pred, labels=groups)

    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=[f'Prédit {g}' for g in groups],
                yticklabels=[f'Réel {g}' for g in groups])
    plt.title('Matrice de confusion', fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()


def display_metrics(y, y_pred):
    print("\n\nRapport de classification :")
    print(classification_report(y, y_pred, target_names=groups))

## Régression logistique

### Pipeline basique

In [ ]:
logistic = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42))
])

logistic.fit(X_train, y_train)
lr_y_pred_test, lr_y_pred_train = train_model_and_detect_overfitting(logistic, y_train, y_test)

# Matrice de confusion et métriques - test
print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE SUR LE TEST\n')
display_confusion_matrix(y_test, lr_y_pred_test)
display_metrics(y_test, lr_y_pred_test)


# Matrice de confusion et métriques - train
print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE SUR LE TRAIN\n')
display_confusion_matrix(y_train, lr_y_pred_train)
display_metrics(y_train, lr_y_pred_train)



Recall = Pourcentage de Vrais Positifs (prédictions correctes) d'une classe parmi l'ensemble des positifs de cette classe.  
Sur le graphique : sur chaque ligne (valeurs réelles), pourcentage du 'prédit correct' par rapport au total de la ligne.  
Autrement dit, sur l'ensemble des communes votant à droite, 98 % sont prédites correctement.


__6 % de recall pour le centre et 15 % de recall pour la gauche : très faible en raison du déséquilibre des données__

### SMOTE

In [ ]:
logistic_smote = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy='all')),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])


# Entraînement
logistic_smote.fit(X_train, y_train)

lr_y_pred_test_smote, lr_y_pred_train_smote = train_model_and_detect_overfitting(logistic_smote, y_train, y_test)


# Matrice de confusion et métriques - test
print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE AVEC SMOTE SUR LE TEST\n')
display_metrics(y_test, lr_y_pred_test_smote)


# Matrice de confusion et métriques - train
print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE AVEC SMOTE SUR LE TRAIN\n')
display_metrics(y_train, lr_y_pred_train_smote)


### Ajustement du poids des classes

In [ ]:
# Modèle avec poids de classes ajustés
logistic_weighted = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, class_weight='balanced'))
])


# Entraînement
logistic_weighted.fit(X_train, y_train)

lr_y_pred_test_weighted, lr_y_pred_train_weighted = train_model_and_detect_overfitting(logistic_weighted, y_train, y_test)


# Matrice de confusion et métriques - test
print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE AVEC WEIGHTED SUR LE TEST\n')
display_metrics(y_test, lr_y_pred_test_weighted)


# Matrice de confusion et métriques - train
print('\n---RÉSULTATS DE LA RÉGRESSION LOGISTIQUE AVEC WEIGHTED SUR LE TRAIN\n')
display_metrics(y_train, lr_y_pred_train_weighted)

## Random Forest

### Pipeline basique

In [ ]:
forest = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42, max_depth=10, min_samples_leaf=5, max_features='sqrt', min_samples_split=15))
])

forest.fit(X_train, y_train)
rf_y_pred_test, rf_y_pred_train = train_model_and_detect_overfitting(forest, y_train, y_test)


# Matrice de confusion et métriques - test
print('\n---RÉSULTATS DE LA RANDOM FOREST SUR LE TEST\n')
display_confusion_matrix(y_test, rf_y_pred_test)
display_metrics(y_test, rf_y_pred_test)


# Matrice de confusion et métriques - train
print('\n---RÉSULTATS DE LA RANDOM FOREST SUR LE TRAIN\n')
display_confusion_matrix(y_train, rf_y_pred_train)
display_metrics(y_train, rf_y_pred_train)


### SMOTE

In [ ]:
forest_smote = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy='all')),
    ('classifier', RandomForestClassifier(n_estimators=200, random_state=42, max_depth=10, min_samples_leaf=5, max_features='sqrt', min_samples_split=15))
])


# Entraînement
forest_smote.fit(X_train, y_train)

rf_y_pred_test_smote, rf_y_pred_train_smote = train_model_and_detect_overfitting(forest_smote, y_train, y_test)


# Métriques - test
print('\n---RÉSULTATS DE LA RANDOM FOREST AVEC SMOTE SUR LE TEST\n')
display_metrics(y_test, rf_y_pred_test_smote)


# Métriques - train
print('\n---RÉSULTATS DE LA RANDOM FOREST AVEC SMOTE SUR LE TRAIN\n')
display_metrics(y_train, rf_y_pred_train_smote)

### Ajustement du poids des classes

In [ ]:
# Modèle avec poids de classes ajustés
forest_weighted = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=200, random_state=42, max_depth=10, class_weight='balanced', min_samples_leaf=5, max_features='sqrt', min_samples_split=15
    ))
])


# Entraînement et détection d'overfitting
rf_y_pred_test_weighted, rf_y_pred_train_weighted = train_model_and_detect_overfitting(forest_weighted, y_train, y_test)


# Matrice de confusion et métriques - test
print('\n---RÉSULTATS DE LA RANDOM FOREST AVEC WEIGHTED SUR LE TEST\n')
display_metrics(y_test, rf_y_pred_test_weighted)


# Matrice de confusion et métriques - train
print('\n---RÉSULTATS DE LA RANDOM FOREST AVEC WEIGHTED SUR LE TRAIN\n')
display_metrics(y_train, rf_y_pred_train_weighted)

## XGBoost

In [ ]:

le = LabelEncoder()

y_train_xgb = le.fit_transform(y_train)
y_test_xgb = le.fit_transform(y_test)



### SMOTE

In [ ]:
xgb_smote = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy='all')),
    ('classifier', xgb.XGBClassifier(
        n_estimators=200,       # Nombre d'arbres
        max_depth=5,           # Profondeur des arbres
        learning_rate=0.1,      # Taux d'apprentissage (η)
        subsample=0.8,          # Échantillonnage des données
        colsample_bytree=0.8,   # Échantillonnage des features
        reg_alpha=0.1,          # Régularisation L1
        reg_lambda=1.0,         # Régularisation L2
        random_state=42,
        n_jobs=-1,
        objective='multi:softprob',
        eval_metric='mlogloss'   # Métrique d'évaluation
    ))
])

xgb_y_pred_test_smote, xgb_y_pred_train_smote = train_model_and_detect_overfitting(xgb_smote, y_train_xgb, y_test_xgb)


# Métriques - test
print('\n---RÉSULTATS DE XGBOOST AVEC SMOTE SUR LE TEST\n')
display_metrics(y_test_xgb, xgb_y_pred_test_smote)


# Métriques - train
print('\n---RÉSULTATS DE XGBOOST AVEC SMOTE SUR LE TRAIN\n')
display_metrics(y_train_xgb, xgb_y_pred_train_smote)

### SMOTEENN

In [ ]:
xgb_smoteen = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTEENN(random_state=42, smote=SMOTE(sampling_strategy='all', random_state=42))),
    ('classifier', xgb.XGBClassifier(
        n_estimators=200,       # Nombre d'arbres
        max_depth=5,           # Profondeur des arbres
        learning_rate=0.1,      # Taux d'apprentissage (η)
        subsample=0.8,          # Échantillonnage des données
        colsample_bytree=0.8,   # Échantillonnage des features
        reg_alpha=0.1,          # Régularisation L1
        reg_lambda=1.0,         # Régularisation L2
        random_state=42,
        n_jobs=-1,
        objective='multi:softprob',
        eval_metric='mlogloss'   # Métrique d'évaluation
    ))
])

xgb_y_pred_test_smoteenn, xgb_y_pred_train_smoteenn = train_model_and_detect_overfitting(xgb_smoteen, y_train_xgb, y_test_xgb)


# Métriques - test
print('\n---RÉSULTATS DE XGBOOST AVEC SMOTEENN SUR LE TEST\n')
display_metrics(y_test_xgb, xgb_y_pred_test_smoteenn)


# Métriques - train
print('\n---RÉSULTATS DE XGBOOST AVEC SMOTEENN SUR LE TRAIN\n')
display_metrics(y_train_xgb, xgb_y_pred_train_smoteenn)


### Class weight

In [ ]:
xgb_weighted = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', xgb.XGBClassifier(
        n_estimators=200,       # Nombre d'arbres
        max_depth=5,           # Profondeur des arbres
        learning_rate=0.1,      # Taux d'apprentissage (η)
        subsample=0.8,          # Échantillonnage des données
        colsample_bytree=0.8,   # Échantillonnage des features
        reg_alpha=0.1,          # Régularisation L1
        reg_lambda=1.0,         # Régularisation L2
        random_state=42,
        n_jobs=-1,
        objective='multi:softprob',
        eval_metric='mlogloss'   # Métrique d'évaluation
    ))
])


weights = compute_sample_weight(class_weight='balanced', y=y_train_xgb)


xgb_weighted.fit(X_train, y_train_xgb, classifier__sample_weight=weights)


train_score = xgb_weighted.score(X_train, y_train_xgb)
test_score = xgb_weighted.score(X_test, y_test_xgb)
gap = abs(train_score - test_score)


print(f"  Train : {train_score:.2%}")
print(f"  Test  : {test_score:.2%}")
print(f"  Gap   : {gap:.2%}\n")


if gap > 0.10:
    print("⚠️ OVERFITTING détecté !")
elif train_score < 0.75:
    print("⚠️ UNDERFITTING détecté !")
else:
    print("✅ Bon équilibre\n")


xgb_y_pred_test_weighted = xgb_weighted.predict(X_test)
xgb_y_pred_train_weighted = xgb_weighted.predict(X_train)

# Métriques - test
print('\n---RÉSULTATS DE XGBOOST WEIGHTED SUR LE TEST\n')
display_metrics(y_test_xgb, xgb_y_pred_test_weighted)


# Métriques - train
print('\n---RÉSULTATS DE XGBOOST WEIGHTED SUR LE TRAIN\n')
display_metrics(y_train_xgb, xgb_y_pred_train_weighted)



# Scripts finaux

1. Créer la table d'entraînement dans la base de données à partir du fichier df_model.csv dans ce dossier

## /training/
1. Appeler la table statistiques et convertir les données pour les insérer dans une table training

In [ ]:
# RÉCUPÉRATION DES DONNÉES DEPUIS LA TABLE STATISTIQUES


# TRANSFORMATION DES DONNÉES BRUTES EN DONNÉES POUR LE MODÈLE

df_train = df_train.drop(columns=['Année','Libellé de la commune', 'Inscrits', 'Abstentions', '% Abs/Ins', 'Votants', '% Vot/Ins', 'Blancs', '% Blancs/Ins', '% Blancs/Vot', 'Nuls', '% Nuls/Ins', '% Nuls/Vot', 'Exprimés', '% Exp/Ins', '% Exp/Vot', '% gauche/Exp', '% centre/Exp', '% droite/Exp'])


def convert_columns_to_percentages(df, list_columns, divider_column):
    for col in list_columns:
        df[col] = df[col] / df[divider_column] * 100
    return df


columns_to_convert = ['Hommes', 'Femmes', 'Agriculteurs', 'Artisans', 'Cadres', 'Intermédiaires', 'Employés', 'Ouvriers', 'Retraités', 'Etudiants', 'Inactifs',
    '15-24 ans', '25-39 ans', '40-54 ans', '55-64 ans', '65-79 ans', '80 ans et +', 'Mariés', 'Pacsés', 'Concubinage', 'Veufs', 'Divorcés', 'Célibataires',
]

df_train = convert_columns_to_percentages(df_train, columns_to_convert, 'Population_active')


columns_to_convert_household = [
    'Personne seule', 'Homme seul', 'Femme seule', 'Colocation', 'Famille', 'Famille monoparentale', 'Couple sans enfant', 'Couple avec enfants',
]
df_train = convert_columns_to_percentages(df_train, columns_to_convert_household, 'Population avec enfants')


# AJOUT DES DONNÉES TRANSFORMÉES DANS LA TABLE TRAINING

# engine = create_engine('')

# df_train.to_sql(
#         name='training',
#         con=engine,
#         if_exists='replace',
#         index=False,
#         chunksize=1000
#     )


2. Récupérer les données de training et appliquer le script ci-dessous (il faudra l'adapter aux données récupérées + au chemin d'enregistrement du modèle joblib) :

In [ ]:
import xgboost as xgb
import joblib
from datetime import date

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# RÉCUPÉRATION DES DONNÉES DEPUIS LA TABLE TRAINING

# split test-train
X = df_train.drop(columns=['Code_INSEE', 'Résultat'])
y = df_train['Résultat']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


# Label encoding pour la variable cible car le modèle ne prend pas de valeurs textuelles
le = LabelEncoder()

y_train_xgb = le.fit_transform(y_train)
y_test_xgb = le.fit_transform(y_test)


# Pipeline XGBoost + SMOTE
xgb_smote = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy='all')),
    ('classifier', xgb.XGBClassifier(
        n_estimators=200,       # Nombre d'arbres
        max_depth=5,           # Profondeur des arbres
        learning_rate=0.1,      # Taux d'apprentissage (η)
        subsample=0.8,          # Échantillonnage des données
        colsample_bytree=0.8,   # Échantillonnage des features
        reg_alpha=0.1,          # Régularisation L1
        reg_lambda=1.0,         # Régularisation L2
        random_state=42,
        n_jobs=-1,
        objective='multi:softprob',
        eval_metric='mlogloss'   # Métrique d'évaluation
    ))
])


# Entraînement
xgb_smote.fit(X_train, y_train_xgb)


# Enregistrement du modèle en joblib
joblib.dump(xgb_smote, f'models/predilection_model_{date.today()}')

## /prediction/

Appliquer le script suivant :


__!!! Attention !!!__ 

Il faudra :
* remplacer l'import du csv par l'appel à la table statistiques en utilisant le code_INSEE de la commune
* gérer les potentielles erreurs si les statistiques sont incomplètes
* remplacer le chemin du fichier lors de l'import joblib

In [ ]:
import pandas as pd
import joblib

df_stats = pd.read_csv('../data/src/insee.csv', sep=';', dtype={'Code_INSEE': str})
df_stats = df_stats.sort_values(by=['Code_INSEE', 'Année'])

# On récupère les données de la commune
df_city = df_stats[(df_stats['Code_INSEE'] == '59001') & ((df_stats['Année'] == 2022) | (df_stats['Année'] == 2011))]

# On crée des projections pour 2027
def calculate_2027_value(column):
    value_2011, value_2022 = column.values
    coeff = (float(value_2022) - float(value_2011)) / 11
    projection_2027 = max(float(value_2022) + (coeff * 5), 0)
    return projection_2027

projections_2027 = df_city.apply(calculate_2027_value)
projections_2027 = projections_2027.drop(labels=['Année', 'Code_INSEE'])

# On convertit les projections en pourcentages pour le modèle
def convert_data_to_percentages(series, list_labels, divider):
    for label in list_labels:
        series[label] = series[label] / series[divider] * 100
    return series


labels_to_convert = ['Hommes', 'Femmes', 'Agriculteurs', 'Artisans', 'Cadres', 'Intermédiaires', 'Employés', 'Ouvriers', 'Retraités', 'Etudiants', 'Inactifs',
    '15-24 ans', '25-39 ans', '40-54 ans', '55-64 ans', '65-79 ans', '80 ans et +', 'Mariés', 'Pacsés', 'Concubinage', 'Veufs', 'Divorcés', 'Célibataires']

projections_2027_rates = projections_2027.copy()
projections_2027_rates = convert_data_to_percentages(projections_2027_rates, labels_to_convert, 'Population_active')


labels_to_convert_household = [
    'Personne seule', 'Homme seul', 'Femme seule', 'Colocation', 'Famille', 'Famille monoparentale', 'Couple sans enfant',
    'Couple avec enfants',
]
projections_2027_rates = convert_data_to_percentages(projections_2027_rates, labels_to_convert_household, 'Population avec enfants')



# Chargement du modèle (adapter le chemin à l'emplacement réel du fichier)
model = joblib.load('models/predilection_model_2026-04-01')


# Prédiction sur les données projetées de la commune
new_data = projections_2027_rates.to_frame().T
prediction = model.predict(new_data)

groups = {0: 'centre', 1: 'droite', 2: 'gauche'}
result = groups[prediction[0]]
